In [2]:
import pandas as pd

In [5]:
folderPath = "/mnt/d/Personal project/DeepLearning/nnForGraphs/attackClassifier/UNSW-NB15/"

In [6]:
dataDF = pd.read_csv(f"{folderPath}UNSW-NB15_2.csv")
dataDF.head()

/tmp/ipykernel_10614/3571369405.py:1: DtypeWarning: Columns (3,39,47) have mixed types. Specify dtype option on import or set low_memory=False.
  dataDF = pd.read_csv(f"{folderPath}UNSW-NB15_2.csv")


,59.166.0.0,6055,149.171.126.5,54145,tcp,FIN,0.072974,4238,60788,31,...,0.6,13,13.1,6,7.1,1,1.1,2,Unnamed: 47,0.7
0,59.166.0.0,7832,149.171.126.3,5607,tcp,FIN,0.144951,5174,91072,31,...,0,13,13,6,7,1,1,2,NaN,0
1,59.166.0.8,11397,149.171.126.6,21,tcp,FIN,0.116107,2934,3742,31,...,1,1,2,7,5,1,1,4,NaN,0
2,59.166.0.0,3804,149.171.126.3,53,udp,CON,0.000986,146,178,31,...,0,13,13,6,7,1,1,2,NaN,0
3,59.166.0.8,14339,149.171.126.6,14724,tcp,FIN,0.038480,8928,320,31,...,0,8,20,7,5,1,1,4,NaN,0
4,59.166.0.8,39094,149.171.126.3,53,udp,CON,0.001026,130,162,31,...,0,8,13,6,5,1,1,1,NaN,0


In [11]:
features = pd.read_csv(f"{folderPath}NUSW-NB15_features.csv" , encoding= "cp1252")

In [12]:
features.head()

,No.,Name,Type,Description
0,1,srcip,nominal,Source IP address
1,2,sport,integer,Source port number
2,3,dstip,nominal,Destination IP address
3,4,dsport,integer,Destination port number
4,5,proto,nominal,Transaction protocol


In [13]:
featuresList = features['Name'].tolist()

In [16]:
dataDF.columns = featuresList

In [17]:
dataDF.columns

Index(['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes',
       'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'Sload', 'Dload',
       'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz',
       'dmeansz', 'trans_depth', 'res_bdy_len', 'Sjit', 'Djit', 'Stime',
       'Ltime', 'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat',
       'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login',
       'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ ltm',
       'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'attack_cat',
       'Label'],
      dtype='object')

In [18]:
required_cols = [
    "srcip",
    "dstip",
    "sport",
    "dsport",
    "proto",
    "state",
    "dur",
    "sbytes",
    "dbytes",
    "Spkts",
    "Dpkts",
    "Sload",
    "Dload",
    "sttl",
    "dttl",
    "Label"
]

dataDF = dataDF[required_cols].copy()

In [19]:
dataDF.head()

,srcip,dstip,sport,dsport,proto,state,dur,sbytes,dbytes,Spkts,Dpkts,Sload,Dload,sttl,dttl,Label
0,59.166.0.0,149.171.126.3,7832,5607,tcp,FIN,0.144951,5174,91072,90,92,2.824127e+05,4.971776e+06,31,29,0
1,59.166.0.8,149.171.126.6,11397,21,tcp,FIN,0.116107,2934,3742,52,54,1.982998e+05,2.530769e+05,31,29,0
2,59.166.0.0,149.171.126.3,3804,53,udp,CON,0.000986,146,178,2,2,5.922921e+05,7.221095e+05,31,29,0
3,59.166.0.8,149.171.126.6,14339,14724,tcp,FIN,0.038480,8928,320,14,6,1.723701e+06,5.550936e+04,31,29,0
4,59.166.0.8,149.171.126.3,39094,53,udp,CON,0.001026,130,162,2,2,5.068226e+05,6.315789e+05,31,29,0


In [20]:
# Convert to lowercase and remove leading/trailing spaces
dataDF['proto'] = (
    dataDF['proto']
    .astype(str)
    .str.strip()
    .str.lower()
)

# Remove internal spaces (e.g., "u dp" -> "udp")
dataDF['proto'] = dataDF['proto'].str.replace(r'\s+', '', regex=True)

In [21]:
# Check for missing values
print(dataDF['proto'].isna().sum())

# Check for empty strings
print((dataDF['proto'] == '').sum())

# Check for leading/trailing whitespace (should be 0)
print((dataDF['proto'] != dataDF['proto'].str.strip()).sum())

0
0
0


In [22]:
dataDF['proto'].value_counts()

proto
tcp         455085
udp         228563
unas          4920
arp           3361
ospf          2215
             ...  
sccopmce        41
ib              41
igmp            10
udt              2
rtp              2
Name: count, Length: 134, dtype: int64

In [23]:
# Remove unwanted spaces in strings
dataDF["proto"] = dataDF["proto"].str.strip()
dataDF["state"] = dataDF["state"].str.strip()

# Encode categorical columns
from sklearn.preprocessing import LabelEncoder

proto_encoder = LabelEncoder()
state_encoder = LabelEncoder()

dataDF["proto"] = proto_encoder.fit_transform(dataDF["proto"])
dataDF["state"] = state_encoder.fit_transform(dataDF["state"])

# Convert numeric columns
numeric_cols = [
    "sport",
    "dsport",
    "dur",
    "sbytes",
    "dbytes",
    "Spkts",
    "Dpkts",
    "Sload",
    "Dload",
    "sttl",
    "dttl"
]

for col in numeric_cols:
    dataDF[col] = pd.to_numeric(dataDF[col], errors="coerce")

dataDF[numeric_cols] = dataDF[numeric_cols].fillna(0)

In [24]:
dataDF.head()

,srcip,dstip,sport,dsport,proto,state,dur,sbytes,dbytes,Spkts,Dpkts,Sload,Dload,sttl,dttl,Label
0,59.166.0.0,149.171.126.3,7832,5607.0,113,5,0.144951,5174,91072,90,92,2.824127e+05,4.971776e+06,31,29,0
1,59.166.0.8,149.171.126.6,11397,21.0,113,5,0.116107,2934,3742,52,54,1.982998e+05,2.530769e+05,31,29,0
2,59.166.0.0,149.171.126.3,3804,53.0,119,2,0.000986,146,178,2,2,5.922921e+05,7.221095e+05,31,29,0
3,59.166.0.8,149.171.126.6,14339,14724.0,113,5,0.038480,8928,320,14,6,1.723701e+06,5.550936e+04,31,29,0
4,59.166.0.8,149.171.126.3,39094,53.0,119,2,0.001026,130,162,2,2,5.068226e+05,6.315789e+05,31,29,0


In [25]:
dataDF.to_csv(f"{folderPath}UNSW-NB15_2_cleaned_encoded.csv")